# Solar Filament Segmentation: train on Kaggle and submit

Pipeline on Kaggle GPUs. All logic lives in the repository
([LucaT2/solar-filament-segmentation](https://github.com/LucaT2/solar-filament-segmentation)); this notebook only
orchestrates it. The same steps run locally with `scripts/run_pipeline.ps1`.

1. **Setup:** clone the code, install missing packages, optionally load earlier results from a Kaggle Dataset.
2. **Train:** fold models (cross-validation) and/or full-data models (all 707 images, no validation), spread over
   the GPUs.
3. **Out-of-fold work:** if folds were trained here, predict their OOF maps, tune post-processing, and fit the
   component classifier. If earlier results were loaded, only re-fit the classifier in this environment.
4. **Submissions:** one CSV per entry in `SUBMISSIONS`, each averaging a different set of models.

### Settings (*Notebook options* in the right-hand panel)
* Accelerator: **GPU T4 ×2**. Internet: **on**.
* *Add Input*: competition **filament-segmentation-2026**, and (optional) your dataset with earlier results, e.g.
  **solar-filament-unet-r34** (created with `scripts/prepare_kaggle_dataset.py`).
* *Add-ons → Secrets*: **`GITHUB_TOKEN`** (read access to the private repo) and, optionally, **`WANDB_API_KEY`**.

In [ ]:
# ---- configuration ---------------------------------------------------------------------------------------
CONFIG = "configs/unet_r34.yaml"      # experiment config inside the repo
OVERRIDES = []                         # config overrides, e.g. ["epochs=40"]
FOLDS = []                             # fold models to train here ([] = reuse the ones from CHECKPOINT_DATASET)
FULL_SEEDS = [42, 43]                  # full-data models (all 707 images, last epoch); one per GPU runs in parallel
CHECKPOINT_DATASET = "solar-filament-unet-r34"   # name of the attached dataset with earlier results (or None)
TTA = True                             # average 4 flips per model at prediction time
SUBMISSIONS = {                        # output name -> models to average (checkpoint names / glob patterns)
    "full": ["full_s*"],               # full-data model(s) only
    "folds_full": ["fold*", "full_s*"],  # 5 fold models + full-data model(s)
    "folds": ["fold*"],                # 5 fold models only: should reproduce the earlier LB 0.37 (sanity check)
}
REPO_URL = "github.com/LucaT2/solar-filament-segmentation.git"
REPO_BRANCH = "main"
REPO_DATASET = None                    # alternative to GITHUB_TOKEN: name of a dataset containing the repo

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess, glob, json
from pathlib import Path

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
try:   # optional W&B tracking; training subprocesses inherit the environment variable
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("W&B tracking enabled")
except Exception:
    os.environ["WANDB_MODE"] = "disabled"
    print("no WANDB_API_KEY secret: tracking disabled")

def find_input(name, marker):
    """Locate an attached dataset by name (Kaggle's mount layout varies), via a file it must contain."""
    hits = [Path(p).parent for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True) if name in p]
    if not hits:
        raise FileNotFoundError(f"attached dataset '{name}' with {marker} not found under /kaggle/input")
    return min(hits, key=lambda p: len(str(p)))

REPO = Path("/kaggle/working/repo")
if REPO_DATASET:
    shutil.copytree(find_input(REPO_DATASET, "pyproject.toml"), REPO, dirs_exist_ok=True)
elif not REPO.exists():
    token = secrets.get_secret("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, f"https://{token}@{REPO_URL}", str(REPO)],
                   check=True, capture_output=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Kaggle images ship torch, numpy, pandas, opencv, scikit-image, scikit-learn, timm and wandb. Install only what is
# missing, pinned to the versions tested locally (requirements.txt), without touching the preinstalled stack.
!pip install -q --no-deps segmentation-models-pytorch==0.5.0 albumentations==2.0.8 albucore==0.0.24 pycocotools
!pip install -q --no-deps -e {REPO}
sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
!python scripts/check_env.py

In [ ]:
from filament.train import TrainConfig
cfg = TrainConfig.load(CONFIG, OVERRIDES)
EXP = Path("/kaggle/working") / cfg.name
EXP.mkdir(parents=True, exist_ok=True)
if CHECKPOINT_DATASET:   # earlier results: fold checkpoints, OOF maps, postprocess.json
    src = find_input(CHECKPOINT_DATASET, "postprocess.json")
    shutil.copytree(src, EXP, dirs_exist_ok=True)
    # `kaggle datasets create --dir-mode zip` uploads sub-folders as zips; Kaggle may or may not extract them
    for archive in EXP.glob("*.zip"):
        if not (EXP / archive.stem).is_dir():
            shutil.unpack_archive(archive, EXP / archive.stem)
        archive.unlink()
    nested = EXP / "oof" / "oof"   # some zips contain the folder itself
    if nested.is_dir():
        for f in nested.iterdir():
            f.rename(EXP / "oof" / f.name)
        nested.rmdir()
    print(f"loaded {src}:", sorted(p.name for p in EXP.glob("*.pt")), "| OOF maps:", len(list((EXP / "oof").glob("*.png"))))
print(cfg)

## 2. Train

Jobs (fold models and full-data models) are spread over the GPUs; each GPU works through its list in one background
process. On a T4 one model takes roughly 60–75 min with the baseline config. Logs: `/kaggle/working/train_gpu<k>.log`.

In [ ]:
jobs = [["--fold", str(k)] for k in FOLDS] + [["--fold", "-1", "--set", f"seed={s}"] for s in FULL_SEEDS]
n_gpu = max(torch.cuda.device_count(), 1)
# two training processes share Kaggle's 4 CPU cores: use 2 data-loading workers each unless overridden
overrides = OVERRIDES + (["num_workers=2"] if n_gpu > 1 and not any(o.startswith("num_workers") for o in OVERRIDES) else [])

def train_cmd(job):
    if "--set" in job:   # merge the job's own overrides (seed) with the global ones
        i = job.index("--set")
        return ["python", "-W", "ignore", "scripts/train.py", "--config", CONFIG, *job[:i], "--set", *overrides, *job[i + 1:]]
    return ["python", "-W", "ignore", "scripts/train.py", "--config", CONFIG, *job] + (["--set", *overrides] if overrides else [])

procs = []
for gpu in range(n_gpu):
    my_jobs = jobs[gpu::n_gpu]
    if not my_jobs:
        continue
    script = " && ".join(subprocess.list2cmdline(train_cmd(j)) for j in my_jobs)
    log = open(f"/kaggle/working/train_gpu{gpu}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    procs.append((subprocess.Popen(["bash", "-c", script], env=env, stdout=log, stderr=subprocess.STDOUT), log, gpu))
    print(f"GPU {gpu}: {my_jobs}")
for p, log, gpu in procs:
    p.wait(); log.close()
    print(f"GPU {gpu} finished with exit code {p.returncode}")
    print("".join(open(f"/kaggle/working/train_gpu{gpu}.log").readlines()[-4:]))
print("checkpoints:", sorted(p.name for p in EXP.glob("*.pt")))

In [ ]:
# Training curves (fold models: validation PQ per threshold; full-data models: loss only)
import pandas as pd
for f in sorted(EXP.glob("log_*.csv")):
    log = pd.read_csv(f)
    pq_cols = [c for c in log.columns if c.startswith("pq@")]
    best = f"best val PQ {log[pq_cols].max(axis=1).max():.4f}" if pq_cols else "no validation"
    print(f"{f.stem}: {len(log)} epochs, {best}, last loss {log.loss.iloc[-1]:.4f}")

## 3. Out-of-fold maps, post-processing, and component classifier

* **Folds trained in this session:** predict their OOF maps with the same TTA as the test set, tune post-processing,
  and fit the component classifier.
* **Folds loaded from the dataset:** keep their OOF maps and `postprocess.json`, and only re-fit the classifier here,
  so it is pickled with this environment's scikit-learn.

In [ ]:
tta_flag = "--tta" if TTA else ""
if FOLDS:
    !python -W ignore scripts/predict.py --exp {cfg.name} {tta_flag} --split oof
    !python -W ignore scripts/tune_postprocess.py --exp {cfg.name} --workers 4 \
        --thresholds 0.4 0.5 0.6 --min-areas 100 200 300 400 500 --min-scores 0
!python -W ignore scripts/fit_rescorer.py --exp {cfg.name}
print(json.dumps(json.loads((EXP / "postprocess.json").read_text()), indent=2))

In [ ]:
# Honest OOF check of the fold models (without the classifier, which was fitted on these same OOF components;
# its honest score is oof_pq_nested in rescorer.json)
trained = sorted(p.stem[4:] for p in EXP.glob("fold*.pt"))
!python -W ignore scripts/make_submission.py --exp {cfg.name} --split oof --no-rescore
!python -W ignore scripts/evaluate.py --pred {EXP}/oof_submission.csv --fold {" ".join(trained)}
print(json.dumps(json.loads((EXP / "rescorer.json").read_text()), indent=2))

## 4. Submissions

For every entry in `SUBMISSIONS`: average the selected models (× 4 flips) on the test images, post-process, apply the
classifier, and write `/kaggle/working/submission_<name>.csv`.

In [ ]:
from filament.paths import get_data_paths
from filament.submission import read_submission, validate_submission
test_stems = [p.stem for p in get_data_paths().test_images.glob("*.jpeg")]
for name, models in SUBMISSIONS.items():
    print(f"\n===== {name}: {models} =====")
    model_args = " ".join(f'"{m}"' for m in models)
    !python -W ignore scripts/predict.py --exp {cfg.name} {tta_flag} --models {model_args} --out test_{name}
    !python -W ignore scripts/make_submission.py --exp {cfg.name} --maps test_{name} --out submission_{name}.csv
    shutil.copy(EXP / f"submission_{name}.csv", f"/kaggle/working/submission_{name}.csv")
    sub = read_submission(f"/kaggle/working/submission_{name}.csv")
    print(f"{len(sub)} filaments on {sub.filament_id.str.rsplit('_', n=1).str[0].nunique()}/{len(test_stems)} images,",
          "validation:", validate_submission(sub, expected_stems=test_stems) or "OK")

## 5. Submit and keep the results

* The CSVs are in this notebook's **Output** (`/kaggle/working/submission_<name>.csv`). Download them and upload on
  the competition page (*Submit Prediction*), or with the Kaggle CLI:
  `kaggle competitions submit -c filament-segmentation-2026 -f submission_full.csv -m "full-data model"`
* New checkpoints (e.g. `full_s42.pt`) are in `/kaggle/working/<name>/`. To reuse them, choose *Save Version →
  Save & Run All* so the output persists; you can then add this notebook's output as an input to a later session.